# 1. Load and inspect the travel bookings
# Replace the catalog and schema with your actual names.

In [0]:
table_name = "/Volumes/damo630_fall_2026/default/multicity_project/multi_city_booking.csv"

bookings_df = spark.read.csv(table_name, header=True, inferSchema=True)

print("Total booking records:", bookings_df.count())

bookings_df.printSchema()

display(bookings_df.limit(10))


## 2. Data Quality Assessment

Before conducting affinity analysis, the dataset
must be evaluated for completeness and consistency.

This stage examines:
- Missing values across all columns
- Number of unique trips and destination cities
- Booking date coverage

These checks help determine whether the data
is suitable for transaction-based analysis.

In [0]:

from pyspark.sql import functions as F

# 2.1 Check missing values across all columns
missing_values = bookings_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in bookings_df.columns
])

print("Missing values by column:")
display(missing_values)

# 2.2 Summarize trips, cities, and booking dates
dataset_summary = bookings_df.agg(
    F.countDistinct("utrip_id").alias("unique_trips"),
    F.countDistinct("city_id").alias("unique_cities"),
    F.min("checkin").alias("earliest_checkin"),
    F.max("checkout").alias("latest_checkout")
)

print("Dataset summary:")
display(dataset_summary)



## 3. Transaction Preparation

The booking-level dataset is transformed into
trip-level transactions for affinity analysis.

Each unique trip ID represents one transaction.
Its basket contains the distinct destination cities
booked within that trip.

Duplicate city visits within the same trip are
counted only once.

In [0]:

from pyspark.sql import functions as F

# 3.1 Check records with invalid trip/city identifiers
quality_checks = bookings_df.agg(
    F.sum(
        (F.col("utrip_id").isNull() |
         (F.trim(F.col("utrip_id")) == "")).cast("int")
    ).alias("invalid_trip_ids"),
    F.sum(
        F.col("city_id").isNull().cast("int")
    ).alias("missing_city_ids"),
    F.sum(
        (F.col("checkout") < F.col("checkin")).cast("int")
    ).alias("invalid_date_order")
)

display(quality_checks)

# 3.2 Keep records with valid transaction identifiers
valid_bookings_df = bookings_df.filter(
    F.col("utrip_id").isNotNull() &
    (F.trim(F.col("utrip_id")) != "") &
    F.col("city_id").isNotNull()
)

# 3.3 Build distinct city baskets
transactions_df = (
    valid_bookings_df
    .groupBy("utrip_id")
    .agg(
        F.array_sort(
            F.collect_set("city_id")
        ).alias("cities")
    )
    .withColumn("basket_size", F.size("cities"))
)

# 3.4 Inspect example baskets without displaying user IDs
display(
    transactions_df
    .select("cities", "basket_size")
    .limit(10)
)

# 3.5 Summarize transaction sizes
basket_summary = transactions_df.agg(
    F.count("*").alias("total_transactions"),
    F.round(F.avg("basket_size"), 2).alias("avg_cities_per_trip"),
    F.min("basket_size").alias("min_cities"),
    F.max("basket_size").alias("max_cities")
)

display(basket_summary)


## 4. Exploratory Data Analysis

### 4.1 Distribution of Cities per Trip

This analysis examines the number of unique destination
cities included in each trip.

The goal is to understand travel patterns and determine
whether the dataset is suitable for association rule mining.

Trips containing only one city cannot produce
city-to-city association rules.

In [0]:

from pyspark.sql import functions as F
import matplotlib.pyplot as plt

# 4.1 Summary statistics
trip_stats = transactions_df.agg(
    F.count("*").alias("total_trips"),
    F.round(F.avg("basket_size"), 2).alias("avg_cities_per_trip"),
    F.max("basket_size").alias("max_cities_per_trip"),
    F.sum(F.when(F.col("basket_size") == 1, 1)
          .otherwise(0)).alias("single_city_trips"),
    F.sum(F.when(F.col("basket_size") >= 2, 1)
          .otherwise(0)).alias("multi_city_trips")
)

display(trip_stats)

# 4.2 Group trip sizes for visualization
# Trips with 11 or more cities are grouped together.

basket_distribution = (
    transactions_df
    .withColumn(
        "size_group",
        F.when(F.col("basket_size") >= 11, 11)
         .otherwise(F.col("basket_size"))
    )
    .groupBy("size_group")
    .count()
    .orderBy("size_group")
)

# Only a small aggregated table is collected to Python
plot_data = basket_distribution.collect()

labels = [
    "11+" if row["size_group"] == 11
    else str(row["size_group"])
    for row in plot_data
]

counts = [row["count"] for row in plot_data]

# 4.3 Visualization
plt.figure(figsize=(10, 5))
plt.bar(labels, counts)

plt.title("Distribution of Unique Cities per Trip")
plt.xlabel("Number of Unique Cities")
plt.ylabel("Number of Trips")

plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


### 4.2 Most Frequently Booked Destination Cities

This analysis identifies the most frequently occurring
destination cities across unique travel transactions.

City popularity is measured by the number of distinct
trips containing each city.

This approach avoids counting repeated bookings of the
same destination within a single trip.

The analysis also examines data sparsity, which can
affect the reliability of frequent itemset discovery.


In [0]:

from pyspark.sql import functions as F
import matplotlib.pyplot as plt

# 4.2.1 Count unique trips containing each city

city_frequency_df = (
    transactions_df
    .select(F.explode("cities").alias("city_id"))
    .groupBy("city_id")
    .agg(F.count("*").alias("trip_count"))
)

total_trips = transactions_df.count()

# Calculate support percentage
city_frequency_df = city_frequency_df.withColumn(
    "support_pct",
    F.round(
        F.col("trip_count") / F.lit(total_trips) * 100,
        3
    )
)

# Identify the top 15 cities
top_15_cities = (
    city_frequency_df
    .orderBy(F.desc("trip_count"), F.asc("city_id"))
    .limit(15)
)

print("Top 15 destination cities:")
display(top_15_cities)


In [0]:

# 4.2.2 Visualize the most frequent cities

top_city_rows = top_15_cities.collect()

city_labels = [
    str(row["city_id"]) for row in top_city_rows
]

trip_counts = [
    row["trip_count"] for row in top_city_rows
]

fig, ax = plt.subplots(figsize=(11, 6))

ax.barh(
    city_labels[::-1],
    trip_counts[::-1]
)

ax.set_title(
    "Top 15 Destination Cities by Unique Trips"
)

ax.set_xlabel("Number of Unique Trips")
ax.set_ylabel("Destination City ID")

ax.grid(axis="x", alpha=0.25)

plt.tight_layout()
plt.show()



### 4.3 Destination Frequency and Data Sparsity

This section evaluates the frequency distribution
of individual destinations.

Cities that appear in very few trips may be
unsuitable for association-rule mining under
a meaningful minimum support threshold.


In [0]:

# 4.3.1 Evaluate destination frequency

sparsity_summary = city_frequency_df.agg(
    F.count("*").alias("total_unique_cities"),

    F.sum(
        F.when(F.col("trip_count") == 1, 1).otherwise(0)
    ).alias("cities_in_one_trip"),

    F.sum(
        F.when(F.col("trip_count") < 10, 1).otherwise(0)
    ).alias("cities_in_fewer_than_10_trips"),

    F.max("trip_count").alias("max_trip_frequency")
)

print("Destination frequency summary:")
display(sparsity_summary)



## 5. Frequent Itemset Mining

### 5.1 Minimum Support Selection

Frequent itemset mining identifies destination cities
and city combinations that appear together in a
sufficient number of travel transactions.

An initial minimum support threshold of 0.1% is used.

This relatively low threshold is appropriate for
exploratory analysis because destination frequencies
are highly sparse.

The threshold will be evaluated based on the number
of frequent itemsets discovered.


In [0]:

from pyspark.sql import functions as F
import math

# 5.1 Define minimum support
min_support = 0.001  # 0.1%

total_trips = transactions_df.count()

min_count = math.ceil(total_trips * min_support)

print("Total transactions:", total_trips)
print("Minimum support:", min_support)
print("Minimum required trip count:", min_count)

# 5.2 Generate frequent 1-itemsets
frequent_1_df = (
    city_frequency_df
    .filter(F.col("trip_count") >= min_count)
    .withColumn(
        "support",
        F.col("trip_count") / F.lit(total_trips)
    )
    .orderBy(F.desc("trip_count"))
)

print("Number of frequent 1-itemsets:", frequent_1_df.count())

display(frequent_1_df.limit(20))



### 5.2 Frequent City Pairs

This stage identifies pairs of destination cities
that appear together within the same trip.

Only cities that satisfy the minimum support
threshold individually are considered as candidates.

Each unordered pair is counted once per trip,
and its support is calculated using the total
number of original transactions.


In [0]:

# 5.3 Keep only cities that meet minimum support
# This reduces the number of candidate pairs.

frequent_city_ids = frequent_1_df.select("city_id")

candidate_baskets_df = (
    transactions_df
    .select(
        "utrip_id",
        F.explode("cities").alias("city_id")
    )
    .join(
        F.broadcast(frequent_city_ids),
        on="city_id",
        how="inner"
    )
    .groupBy("utrip_id")
    .agg(
        F.array_sort(
            F.collect_set("city_id")
        ).alias("cities")
    )
    .filter(F.size("cities") >= 2)
)

# 5.4 Generate unique unordered city pairs
candidate_pairs_df = (
    candidate_baskets_df
    .select(
        "utrip_id",
        "cities",
        F.posexplode("cities").alias("i", "city_a")
    )
    .select(
        "utrip_id",
        "i",
        "city_a",
        F.posexplode("cities").alias("j", "city_b")
    )
    .filter(F.col("i") < F.col("j"))
)

# 5.5 Count the frequency of each city pair
frequent_2_df = (
    candidate_pairs_df
    .groupBy("city_a", "city_b")
    .agg(F.count("*").alias("trip_count"))
    .filter(F.col("trip_count") >= min_count)
    .withColumn(
        "support",
        F.col("trip_count") / F.lit(total_trips)
    )
    .orderBy(F.desc("trip_count"))
)

print("Number of frequent 2-itemsets:", frequent_2_df.count())

print("Top 20 frequent city pairs:")
display(frequent_2_df.limit(20))
